# 10 — Affinity and Complex-Stability Scoring

**Purpose:** Score all Boltz-2-validated PD-L1 mini-binder designs using static structural metrics, physics-based affinity proxies, and MD-derived dynamic stability features. Produce a consensus scorecard that ranks designs for experimental prioritization.

**Scientific question:** Given minimized structures and MD trajectories, can we produce relative affinity-like and stability-like scores for modeled complexes — and do static scoring functions agree with dynamic stability metrics?

**Key distinction from previous notebooks:**
- Notebook 07 performed pre-relaxation interface fingerprinting on threaded full-sidechain complexes.
- Notebook 08 performed energy minimization and short MD on 3 representative complexes.
- Notebook 09 validated all 31 designs with Boltz-2 independent complex prediction.
- **Notebook 10 systematically simulates and scores all 28 Boltz-validated designs**, producing the first complete comparative ranking across the design set.

**What this notebook does NOT claim:** These scores are not experimental affinities. They are comparative triage features that ask whether modeled complexes retain interfaces after minimization and short MD, and whether static scoring functions agree with dynamic stability metrics. PRODIGY ΔG estimates are trained on natural complexes and may not generalize to de novo designed interfaces. All metrics are proxies; absolute values should not be trusted.

**Inputs:**
- 31 threaded complex PDBs from NB07 (in `portfolio_pages/Page3_.../passed_31/`)
- Boltz-2 tier classifications from NB09 (28 validated, 3 uncertain)
- NB07 fingerprinting summary JSON
- 3 existing MD trajectories from NB08 (reused, not re-simulated)

**Outputs:**
- `10_static_scores.csv` — post-minimization structural metrics for all designs
- `10_md_stability_scores.csv` — MD-derived dynamic stability metrics
- `10_prodigy_scores.csv` — PRODIGY ΔG and Kd estimates
- `10_complex_scorecard.csv` — unified scorecard with all features and evidence flags
- `10_consensus_ranking.csv` — final ranked list
- `10_scoring_summary.json` — structured summary for portfolio page

**Compute:** OpenMM on Colab T4 GPU for MD (~1.5 hr/design × 25 new designs ≈ 38 T4-hours). PRODIGY and static scoring are CPU-only. Restart-safe throughout.


## Metric provenance

| Metric | Source | Direction | Interpretation | Caveat |
|--------|--------|-----------|---------------|--------|
| BSA (Å²) | Static minimized structure | Larger = better | Buried interface area; larger suggests more extensive interface | Does not guarantee specificity or affinity |
| Residue contact pairs | Static minimized structure | Larger = better | Number of unique inter-chain residue pairs within 4.5 Å | Topological proximity, not binding energy |
| Polar close-contacts | Static minimized structure | Larger = better | Polar heavy-atom pairs (N/O/S) within 3.5 Å | Distance-only proxy — no angle geometry, not validated H-bonds |
| PRODIGY ΔG (kcal/mol) | PRODIGY model on minimized structure | More negative = better | Empirical binding affinity proxy (r ≈ 0.73 on benchmark) | Trained on natural complexes; performance on de novo designs is unknown |
| PRODIGY Kd (M) | PRODIGY model on minimized structure | Lower = better | Dissociation constant estimate | Same caveats as ΔG |
| Boltz-2 ipTM | Boltz-2 complex prediction (NB09) | Higher = better | Independent fold confidence for the complex | Prediction confidence, not binding affinity |
| Binder RMSD (Å) | MD trajectory | Lower = better | Binder backbone stability during simulation | Short (1 ns) trajectories are not definitive |
| Target RMSD (Å) | MD trajectory | Lower = better | Target stability (should be low for well-behaved simulations) | Validates simulation health, not binding |
| COM drift (Å) | MD trajectory | Closer to 0 = better | Change in binder-target center-of-mass distance | Positive drift suggests dissociation tendency |
| Contact count (atoms) | MD trajectory | Higher = better | Binder heavy atoms with ≥1 target neighbor within 4.5 Å per frame | Atom-level metric; not directly comparable to static residue-pair contacts |
| Binder RMSF (Å) | MD trajectory | Lower = better | Per-residue flexibility of binder Cα atoms | High RMSF may indicate disorder or insufficient sampling |


In [1]:
# ============================================================
# Cell 0 — Colab setup: install dependencies, verify GPU
# ============================================================
#
# Dependencies: None (first cell)
# External files: None
# Compute: CPU for install; GPU required for MD cells later
# Purpose:
#   1. Install OpenMM, MDTraj, PDBFixer, PRODIGY, BioPython
#   2. Verify GPU availability
#   3. Set up restart-safe sentinel
#
# NOTE: Run this cell first. After first install, Colab may
# require a runtime restart — rerun this cell after restart.
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
SETUP_SENTINEL = Path('/content/.nb10_setup_complete') if IN_COLAB else Path('.nb10_setup_complete')

print(f'Running in Colab: {IN_COLAB}')
print(f'Python: {sys.version.split()[0]}')

if not SETUP_SENTINEL.exists():
    print('Installing dependencies...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'openmm', 'pdbfixer', 'mdtraj', 'biopython',
        'prodigy-prot', 'scipy', 'pandas', 'numpy',
    ])
    SETUP_SENTINEL.write_text('ok\n')
    if IN_COLAB:
        print('\nDependencies installed. Restarting runtime...')
        try:
            from google.colab import runtime
            runtime.restart_runtime()
        except Exception:
            os.kill(os.getpid(), 9)
    else:
        raise SystemExit('Restart kernel, then rerun Cell 0.')

# Post-restart verification
print('Verifying imports...')
import numpy as np
import pandas as pd
import mdtraj as md
import openmm
import openmm.app as app
import openmm.unit as unit
from pdbfixer import PDBFixer
from Bio.PDB import PDBParser

print(f'numpy:   {np.__version__}')
print(f'pandas:  {pd.__version__}')
print(f'mdtraj:  {md.__version__}')
print(f'openmm:  {openmm.__version__}')

# GPU check
import openmm as mm
platforms = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print(f'OpenMM platforms: {platforms}')
if 'CUDA' in platforms:
    print('GPU: CUDA available')
elif 'OpenCL' in platforms:
    print('GPU: OpenCL available (slower than CUDA)')
else:
    print('WARNING: No GPU platform found. MD will be very slow on CPU.')

print('\n✓ Cell 0 complete')


Running in Colab: True
Python: 3.12.13
Verifying imports...
numpy:   2.0.2
pandas:  2.2.2
mdtraj:  1.11.1.post1
openmm:  8.5.2
OpenMM platforms: ['Reference', 'CPU', 'OpenCL']
GPU: OpenCL available (slower than CUDA)

✓ Cell 0 complete


In [2]:
# ============================================================
# Cell 1 — Mount Drive, set paths, build master design table
# ============================================================
#
# Dependencies: Cell 0
# External files:
#   - portfolio_pages/Page3_.../passed_31/*.pdb (threaded complexes)
#   - data/results/boltz_complex_validation/09_boltz_tier_classification.csv
#   - data/results/fingerprinting_param_sensitivity/fingerprint_summary_full_sidechain.json
#   - data/results/md_simulations_threaded_complexes/md_manifest.csv
# Compute: CPU only
# Purpose: Load all upstream data, merge into a single master table,
#          filter to Boltz-validated designs, locate PDB paths
# Outputs: df_master, df_validated, DESIGN_IDS, path constants
# ============================================================

import json
from pathlib import Path

# ── Mount Google Drive ──
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ── Project paths ──
PROJECT_ROOT = Path('/content/drive/Othercomputers/My Mac/pdl1-mini-binder-design')
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'

# Input directories
COMPLEXES_DIR = PROJECT_ROOT / 'portfolio_pages' / 'Page3_interaction_fingerprinting' / 'portfolio_page3-viewer_structures' / 'passed_31'
BOLTZ_DIR = PROJECT_ROOT / 'data' / 'results' / 'boltz_complex_validation'
FP_DIR = PROJECT_ROOT / 'data' / 'results' / 'fingerprinting_param_sensitivity'
EXISTING_MD_DIR = PROJECT_ROOT / 'data' / 'results' / 'md_simulations_threaded_complexes'

# Output directory
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'results' / 'affinity_scoring'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Working directory (local, fast)
WORK_DIR = Path('/content/nb10_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Complexes dir: {COMPLEXES_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

# ── Verify input directories exist ──
for d, label in [(COMPLEXES_DIR, 'Threaded complexes'),
                 (BOLTZ_DIR, 'Boltz-2 results'),
                 (FP_DIR, 'Fingerprinting results')]:
    assert d.exists(), f'{label} directory not found: {d}'
    print(f'  ✓ {label}: {d.name}')

# ── Discover threaded complex PDBs ──
complex_pdbs = sorted(COMPLEXES_DIR.glob('*_complex.pdb'))
binder_pdbs = sorted(COMPLEXES_DIR.glob('*_fold_binder.pdb'))
print(f'\nFound {len(complex_pdbs)} complex PDBs, {len(binder_pdbs)} binder PDBs')

# Build a lookup: design_name -> complex PDB path
# Filenames like: 01_len70_clusterA_noise05__design_19_0__design_19_0_rank0__threaded_complex.pdb
# Boltz CSV names: len70_clusterA_noise05__design_19_0_rank0
# The filename has a doubled __design_X_0__design_X_0_rankN pattern that must be collapsed.
import re

complex_lookup = {}
for p in complex_pdbs:
    name = p.stem
    # Strip leading rank number (e.g., '01_')
    parts = name.split('_', 1)
    if len(parts) == 2 and parts[0].isdigit():
        key = parts[1]
    else:
        key = name
    # Strip trailing __threaded_complex
    key = key.replace('__threaded_complex', '')
    # Collapse doubled design ID: __design_X_0__design_X_0_rankN -> __design_X_0_rankN
    key = re.sub(r'(__design_\d+_\d+)__design_\d+_\d+_(rank)', r'\1_\2', key)
    complex_lookup[key] = str(p)

print(f'Parsed {len(complex_lookup)} design keys from filenames')
for k in list(complex_lookup.keys())[:3]:
    print(f'  {k}')

# ── Load Boltz-2 tier classifications ──
boltz_csv = BOLTZ_DIR / '09_boltz_tier_classification.csv'
assert boltz_csv.exists(), f'Boltz tier CSV not found: {boltz_csv}'
df_boltz = pd.read_csv(boltz_csv)
print(f'\nBoltz tiers: {df_boltz["boltz_tier"].value_counts().to_dict()}')

# Verify design name overlap between PDB filenames and Boltz CSV
boltz_names = set(df_boltz['design_name'].values)
overlap = boltz_names & set(complex_lookup.keys())
print(f'PDB-Boltz name overlap: {len(overlap)}/{len(boltz_names)}')
if len(overlap) < len(boltz_names):
    missing = boltz_names - set(complex_lookup.keys())
    print(f'WARNING: {len(missing)} unmatched Boltz names')
    for m in sorted(missing)[:5]:
        print(f'  {m}')

# ── Load fingerprinting summary ──
fp_json = FP_DIR / 'fingerprint_summary_full_sidechain.json'
if fp_json.exists():
    with open(fp_json) as f:
        fp_summary = json.load(f)
    print(f'Fingerprinting summary loaded: {fp_summary["n_designs_analyzed"]} designs')
else:
    print(f'WARNING: Fingerprinting summary not found at {fp_json}')
    fp_summary = None

# ── Check existing MD results ──
existing_md_manifest = EXISTING_MD_DIR / 'md_manifest.csv'
if existing_md_manifest.exists():
    df_existing_md = pd.read_csv(existing_md_manifest)
    existing_md_designs = set(df_existing_md['design_name'].values) if 'design_name' in df_existing_md.columns else set()
    print(f'\nExisting MD simulations found: {len(existing_md_designs)}')
    for d in sorted(existing_md_designs):
        print(f'  {d}')
else:
    existing_md_designs = set()
    print('No existing MD manifest found')

# ── Build master design table ──
df_master = df_boltz.copy()
df_master['complex_pdb'] = df_master['design_name'].map(complex_lookup)
n_matched = df_master['complex_pdb'].notna().sum()
print(f'\nPDB path matching: {n_matched}/{len(df_master)} designs matched')

df_master['has_existing_md'] = df_master['design_name'].isin(existing_md_designs)

if fp_summary and 'alignment_rmsds' in fp_summary:
    df_master['fp_alignment_rmsd'] = df_master['design_name'].map(fp_summary['alignment_rmsds'])

# ── Filter to simulation candidates ──
df_validated = df_master[
    (df_master['boltz_tier'] == 'validated') &
    (df_master['complex_pdb'].notna())
].copy()

df_needs_md = df_validated[~df_validated['has_existing_md']].copy()
df_has_md = df_validated[df_validated['has_existing_md']].copy()

print(f'\n=== Simulation plan ===')
print(f'Total Boltz-validated designs: {len(df_validated)}')
print(f'Already have MD trajectories:  {len(df_has_md)}')
print(f'Need new MD simulations:       {len(df_needs_md)}')
print(f'Estimated GPU time for new:    ~{len(df_needs_md) * 1.5:.0f} T4-hours')

ALL_DESIGN_IDS = df_validated['design_name'].tolist()
NEW_MD_DESIGN_IDS = df_needs_md['design_name'].tolist()

print(f'\n✓ Cell 1 complete')


Mounted at /content/drive
Project root: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design
Complexes dir: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/portfolio_pages/Page3_interaction_fingerprinting/portfolio_page3-viewer_structures/passed_31
Output dir: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/affinity_scoring
  ✓ Threaded complexes: passed_31
  ✓ Boltz-2 results: boltz_complex_validation
  ✓ Fingerprinting results: fingerprinting_param_sensitivity

Found 31 complex PDBs, 0 binder PDBs
Parsed 31 design keys from filenames
  len70_clusterA_noise05__design_19_0_rank0
  len70_distributed_noise0__design_17_0_rank1
  len25_clusterA_noise0__design_11_0_rank1

Boltz tiers: {'validated': 28, 'uncertain': 3}
PDB-Boltz name overlap: 31/31

Existing MD simulations found: 3
  len70_clusterA_noise05__design_19_0_rank0
  len70_clusterB_noise0__design_19_0_rank1
  len70_distributed_noise0__design_17_0_rank1

PDB path matching: 31/31 designs

In [3]:
# ============================================================
# Cell 2 — Input validation: check PDB integrity before simulation
# ============================================================
#
# Dependencies: Cell 0, Cell 1
# External files: Complex PDBs from Cell 1
# Compute: CPU only
# Purpose: Validate every complex PDB before committing GPU time.
#   Checks: chain count, residue count, backbone completeness,
#   PDBFixer parseability. Flag issues rather than fail silently.
# Outputs: df_validation (DataFrame with pass/fail per design)
# ============================================================

from Bio.PDB import PDBParser
from pdbfixer import PDBFixer
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='Bio')

parser = PDBParser(QUIET=True)

validation_records = []

for _, row in df_validated.iterrows():
    design_name = row['design_name']
    pdb_path = row['complex_pdb']
    rec = {'design_name': design_name, 'pdb_path': pdb_path}

    try:
        structure = parser.get_structure(design_name, pdb_path)
        model = structure[0]
        chains = list(model.get_chains())
        rec['n_chains'] = len(chains)
        rec['chain_ids'] = ','.join(c.id for c in chains)

        for c in chains:
            residues = [r for r in c.get_residues() if r.id[0] == ' ']
            rec[f'chain_{c.id}_residues'] = len(residues)

            missing_bb = 0
            for r in residues:
                atom_names = {a.name for a in r.get_atoms()}
                if not {'N', 'CA', 'C'}.issubset(atom_names):
                    missing_bb += 1
            rec[f'chain_{c.id}_missing_backbone'] = missing_bb

        fixer = PDBFixer(filename=str(pdb_path))
        fixer.findMissingResidues()
        fixer.findMissingAtoms()
        n_missing_res = sum(len(v) for v in fixer.missingResidues.values())
        n_missing_atoms = sum(len(v) for v in fixer.missingAtoms.values())
        rec['missing_residues'] = n_missing_res
        rec['missing_atoms'] = n_missing_atoms
        rec['pdbfixer_ok'] = True

        issues = []
        if rec['n_chains'] != 2:
            issues.append(f'expected 2 chains, found {rec["n_chains"]}')
        if n_missing_res > 0:
            issues.append(f'{n_missing_res} missing residues')
        for c in chains:
            mb = rec.get(f'chain_{c.id}_missing_backbone', 0)
            if mb > 0:
                issues.append(f'chain {c.id}: {mb} residues missing backbone atoms')

        rec['issues'] = '; '.join(issues) if issues else 'none'
        rec['validation_pass'] = len(issues) == 0

    except Exception as e:
        rec['issues'] = f'parse error: {str(e)}'
        rec['validation_pass'] = False
        rec['pdbfixer_ok'] = False

    validation_records.append(rec)

df_validation = pd.DataFrame(validation_records)

n_pass = df_validation['validation_pass'].sum()
n_fail = (~df_validation['validation_pass']).sum()

print(f'=== Input validation ===')
print(f'Checked: {len(df_validation)} designs')
print(f'Pass:    {n_pass}')
print(f'Fail:    {n_fail}')

if n_fail > 0:
    print(f'\n--- Failed designs ---')
    for _, row in df_validation[~df_validation['validation_pass']].iterrows():
        print(f"  {row['design_name']}: {row['issues']}")

print(f'\n--- Sample (first 3 passing) ---')
for _, row in df_validation[df_validation['validation_pass']].head(3).iterrows():
    print(f"  {row['design_name']}: chains={row['chain_ids']}, "
          f"missing_atoms={row.get('missing_atoms', '?')}")

valid_designs = set(df_validation[df_validation['validation_pass']]['design_name'])
SIM_DESIGN_IDS = [d for d in NEW_MD_DESIGN_IDS if d in valid_designs]
print(f'\nDesigns cleared for simulation: {len(SIM_DESIGN_IDS)}')

val_csv = OUTPUT_DIR / '10_input_validation.csv'
df_validation.to_csv(val_csv, index=False)
print(f'✓ Saved: {val_csv}')

print(f'\n✓ Cell 2 complete')


=== Input validation ===
Checked: 28 designs
Pass:    28
Fail:    0

--- Sample (first 3 passing) ---
  len70_clusterA_noise05__design_19_0_rank0: chains=B,A, missing_atoms=654
  len70_distributed_noise0__design_17_0_rank1: chains=B,A, missing_atoms=618
  len25_clusterA_noise0__design_11_0_rank1: chains=B,A, missing_atoms=513

Designs cleared for simulation: 25
✓ Saved: /content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/results/affinity_scoring/10_input_validation.csv

✓ Cell 2 complete


In [4]:
# ============================================================
# Cell 3 — System preparation: PDBFixer cleanup + solvation
# ============================================================
#
# Dependencies: Cell 0, Cell 1, Cell 2
# External files: Complex PDBs (validated in Cell 2)
# Compute: CPU only (no GPU needed)
# Purpose: For each design that needs new MD:
#   1. PDBFixer: add missing atoms, hydrogens at pH 7.0
#   2. Solvate: AMBER14/TIP3P, 150 mM NaCl, 1.0 nm padding
#   3. Write solvated PDB to Drive
# Restart-safe: skips designs with existing solvated.pdb
# Outputs: prep_results (list of dicts), solvated PDBs on Drive
#
# Water model consistency (NB08 review fix):
#   ForceField uses 'amber14/tip3p.xml' (not tip3pfb.xml)
#   addSolvent uses model='tip3p'
# ============================================================

from openmm.app import PDBFile, Modeller, ForceField
from pdbfixer import PDBFixer
import openmm.unit as unit
import time

FORCEFIELD_FILES = ['amber14-all.xml', 'amber14/tip3p.xml']
WATER_MODEL = 'tip3p'
PADDING = 1.0 * unit.nanometers
IONIC_STRENGTH = 0.15 * unit.molar

prep_results = []
prep_errors = []

print(f'Preparing {len(SIM_DESIGN_IDS)} designs for simulation...\n')

for i, design_name in enumerate(SIM_DESIGN_IDS):
    design_dir = OUTPUT_DIR / design_name
    design_dir.mkdir(parents=True, exist_ok=True)
    solvated_pdb = design_dir / 'solvated.pdb'

    if solvated_pdb.exists():
        print(f'[{i+1}/{len(SIM_DESIGN_IDS)}] {design_name}: solvated.pdb exists, skipping')
        prep_results.append({
            'design_name': design_name,
            'status': 'cached',
            'solvated_pdb': str(solvated_pdb),
        })
        continue

    pdb_path = df_validated[df_validated['design_name'] == design_name]['complex_pdb'].values[0]
    t0 = time.time()

    try:
        print(f'[{i+1}/{len(SIM_DESIGN_IDS)}] Preparing: {design_name}')

        fixer = PDBFixer(filename=str(pdb_path))
        fixer.findMissingResidues()
        fixer.findMissingAtoms()
        fixer.findNonstandardResidues()
        fixer.replaceNonstandardResidues()
        fixer.addMissingAtoms()
        fixer.addMissingHydrogens(pH=7.0)

        forcefield = ForceField(*FORCEFIELD_FILES)
        modeller = Modeller(fixer.topology, fixer.positions)
        modeller.addSolvent(
            forcefield,
            model=WATER_MODEL,
            padding=PADDING,
            ionicStrength=IONIC_STRENGTH,
            positiveIon='Na+',
            negativeIon='Cl-',
        )

        n_atoms = modeller.topology.getNumAtoms()
        n_residues = modeller.topology.getNumResidues()

        with open(solvated_pdb, 'w') as f:
            PDBFile.writeFile(modeller.topology, modeller.positions, f)

        elapsed = time.time() - t0
        print(f'  ✓ {n_atoms} atoms, {n_residues} residues ({elapsed:.1f}s)')

        prep_results.append({
            'design_name': design_name,
            'status': 'prepared',
            'solvated_pdb': str(solvated_pdb),
            'n_atoms': n_atoms,
            'n_residues': n_residues,
            'prep_time_s': round(elapsed, 1),
        })

    except Exception as e:
        elapsed = time.time() - t0
        print(f'  ✗ FAILED: {e} ({elapsed:.1f}s)')
        prep_errors.append({'design_name': design_name, 'error': str(e)})

print(f'\n=== Preparation summary ===')
print(f'Prepared: {sum(1 for r in prep_results if r["status"] == "prepared")}')
print(f'Cached:   {sum(1 for r in prep_results if r["status"] == "cached")}')
print(f'Failed:   {len(prep_errors)}')

if prep_errors:
    print('\n--- Errors ---')
    for e in prep_errors:
        print(f"  {e['design_name']}: {e['error']}")

print(f'\n✓ Cell 3 complete')


Preparing 25 designs for simulation...

[1/25] len25_clusterA_noise0__design_11_0_rank1: solvated.pdb exists, skipping
[2/25] len70_clusterA_noise0__design_3_0_rank7: solvated.pdb exists, skipping
[3/25] len70_distributed_noise0__design_5_0_rank2: solvated.pdb exists, skipping
[4/25] len70_clusterB_noise0__design_17_0_rank0: solvated.pdb exists, skipping
[5/25] len100_clusterA_noise0__design_16_0_rank6: solvated.pdb exists, skipping
[6/25] len100_distributed_noise0__design_5_0_rank0: solvated.pdb exists, skipping
[7/25] len70_clusterA_noise0__design_16_0_rank5: solvated.pdb exists, skipping
[8/25] len25_clusterA_noise0__design_15_0_rank0: solvated.pdb exists, skipping
[9/25] len100_clusterA_noise0__design_13_0_rank4: solvated.pdb exists, skipping
[10/25] len100_clusterB_noise05__design_0_0_rank1: solvated.pdb exists, skipping
[11/25] len70_clusterB_noise0__design_15_0_rank2: solvated.pdb exists, skipping
[12/25] len70_distributed_noise0__design_10_0_rank3: solvated.pdb exists, skipping

In [5]:
# ============================================================
# Cell 4 — Energy minimization
# ============================================================
#
# Dependencies: Cell 0, Cell 1, Cell 2, Cell 3
# External files: solvated.pdb from Cell 3 (on Drive)
# Compute: GPU (CUDA preferred, mixed precision)
# Purpose: Remove steric clashes from threaded structures.
#   Writes minimized.pdb for each design.
# Restart-safe: skips designs with existing minimized.pdb
# Outputs: minimized PDBs on Drive, min_results list
# ============================================================

import openmm as mm
from openmm.app import PDBFile, ForceField, Simulation
from openmm.app import PME, HBonds
import openmm.unit as unit
import time

# Select best available platform with explicit precision
platform = None
platform_properties = {}
for pname in ['CUDA', 'OpenCL', 'CPU']:
    try:
        platform = mm.Platform.getPlatformByName(pname)
        if pname == 'CUDA':
            platform_properties = {'Precision': 'mixed'}
        elif pname == 'OpenCL':
            platform_properties = {'Precision': 'mixed'}
        else:
            platform_properties = {}
        print(f'Using platform: {pname}'
              f'{" (mixed precision)" if platform_properties else ""}')
        break
    except Exception:
        continue

FORCEFIELD_FILES = ['amber14-all.xml', 'amber14/tip3p.xml']

min_results = []
designs_to_minimize = [r for r in prep_results if r['status'] in ('prepared', 'cached')]

print(f'Minimizing {len(designs_to_minimize)} designs...\n')

for i, prep in enumerate(designs_to_minimize):
    design_name = prep['design_name']
    design_dir = OUTPUT_DIR / design_name
    minimized_pdb = design_dir / 'minimized.pdb'

    if minimized_pdb.exists():
        print(f'[{i+1}/{len(designs_to_minimize)}] {design_name}: minimized.pdb exists, skipping')
        min_results.append({'design_name': design_name, 'status': 'cached'})
        continue

    solvated_pdb = prep['solvated_pdb']
    t0 = time.time()

    try:
        print(f'[{i+1}/{len(designs_to_minimize)}] Minimizing: {design_name}')

        pdb = PDBFile(str(solvated_pdb))
        forcefield = ForceField(*FORCEFIELD_FILES)
        system = forcefield.createSystem(
            pdb.topology,
            nonbondedMethod=PME,
            nonbondedCutoff=1.0 * unit.nanometers,
            constraints=HBonds,
        )

        integrator = mm.LangevinMiddleIntegrator(
            300 * unit.kelvin, 1 / unit.picosecond, 0.002 * unit.picoseconds
        )

        simulation = Simulation(pdb.topology, system, integrator,
                                platform, platform_properties)
        simulation.context.setPositions(pdb.positions)

        state0 = simulation.context.getState(getEnergy=True)
        e0 = state0.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)

        simulation.minimizeEnergy(maxIterations=5000)

        state1 = simulation.context.getState(getEnergy=True, getPositions=True)
        e1 = state1.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
        positions = state1.getPositions()

        with open(minimized_pdb, 'w') as f:
            PDBFile.writeFile(pdb.topology, positions, f)

        elapsed = time.time() - t0
        print(f'  ✓ E: {e0:.0f} → {e1:.0f} kJ/mol ({elapsed:.1f}s)')

        min_results.append({
            'design_name': design_name,
            'status': 'minimized',
            'energy_before_kJmol': round(e0, 1),
            'energy_after_kJmol': round(e1, 1),
            'min_time_s': round(elapsed, 1),
        })

    except Exception as e:
        elapsed = time.time() - t0
        print(f'  ✗ FAILED: {e} ({elapsed:.1f}s)')
        min_results.append({'design_name': design_name, 'status': 'failed', 'error': str(e)})

n_min = sum(1 for r in min_results if r['status'] in ('minimized', 'cached'))
n_fail = sum(1 for r in min_results if r['status'] == 'failed')
print(f'\nMinimization: {n_min} succeeded, {n_fail} failed')

print(f'\n✓ Cell 4 complete')


Using platform: OpenCL (mixed precision)
Minimizing 25 designs...

[1/25] len25_clusterA_noise0__design_11_0_rank1: minimized.pdb exists, skipping
[2/25] len70_clusterA_noise0__design_3_0_rank7: minimized.pdb exists, skipping
[3/25] len70_distributed_noise0__design_5_0_rank2: minimized.pdb exists, skipping
[4/25] len70_clusterB_noise0__design_17_0_rank0: minimized.pdb exists, skipping
[5/25] len100_clusterA_noise0__design_16_0_rank6: minimized.pdb exists, skipping
[6/25] len100_distributed_noise0__design_5_0_rank0: minimized.pdb exists, skipping
[7/25] len70_clusterA_noise0__design_16_0_rank5: minimized.pdb exists, skipping
[8/25] len25_clusterA_noise0__design_15_0_rank0: minimized.pdb exists, skipping
[9/25] len100_clusterA_noise0__design_13_0_rank4: minimized.pdb exists, skipping
[10/25] len100_clusterB_noise05__design_0_0_rank1: minimized.pdb exists, skipping
[11/25] len70_clusterB_noise0__design_15_0_rank2: minimized.pdb exists, skipping
[12/25] len70_distributed_noise0__design_10_

In [6]:
# ============================================================
# Cell 5 — Equilibration: NVT 100 ps → NPT 200 ps
# ============================================================
#
# Dependencies: Cell 0, Cell 1, Cell 2, Cell 3, Cell 4
# External files: minimized.pdb from Cell 4 (on Drive)
# Compute: GPU (CUDA preferred, mixed precision)
# Purpose: Equilibrate solvent and temperature around the
#   minimized protein complex with position restraints.
#   NVT phase: heat to 300K with restrained protein.
#   NPT phase: equilibrate density at 1 atm.
# Restart-safe: skips designs with existing equilibrated.pdb
# Outputs: equilibrated PDBs + state XML on Drive
#
# Position restraint energy expression (NB08 review fix):
#   0.5 * k * periodicdistance(x,y,z,x0,y0,z0)^2
# ============================================================

import openmm as mm
from openmm.app import PDBFile, ForceField, Simulation, StateDataReporter
from openmm.app import PME, HBonds
import openmm.unit as unit
import time

FORCEFIELD_FILES = ['amber14-all.xml', 'amber14/tip3p.xml']
NVT_STEPS = 50000   # 100 ps at 2 fs
NPT_STEPS = 100000  # 200 ps at 2 fs
RESTRAINT_K = 1000.0  # kJ/mol/nm^2
TIMESTEP = 0.002 * unit.picoseconds


def add_position_restraints(system, topology, positions, k=RESTRAINT_K):
    """Add position restraints on non-hydrogen protein atoms.
    Uses corrected energy expression with 0.5 factor."""
    restraint = mm.CustomExternalForce(
        '0.5*k*periodicdistance(x, y, z, x0, y0, z0)^2'
    )
    restraint.addGlobalParameter('k', k * unit.kilojoules_per_mole / unit.nanometers**2)
    restraint.addPerParticleParameter('x0')
    restraint.addPerParticleParameter('y0')
    restraint.addPerParticleParameter('z0')

    for atom in topology.atoms():
        if atom.residue.name not in ('HOH', 'NA', 'CL') and atom.element.symbol != 'H':
            pos = positions[atom.index]
            restraint.addParticle(atom.index, [
                pos[0].value_in_unit(unit.nanometers),
                pos[1].value_in_unit(unit.nanometers),
                pos[2].value_in_unit(unit.nanometers),
            ])

    system.addForce(restraint)
    return restraint


designs_to_equil = [r for r in min_results if r['status'] in ('minimized', 'cached')]
equil_results = []

print(f'Equilibrating {len(designs_to_equil)} designs...\n')

for i, rec in enumerate(designs_to_equil):
    design_name = rec['design_name']
    design_dir = OUTPUT_DIR / design_name
    equil_pdb = design_dir / 'equilibrated.pdb'
    equil_state = design_dir / 'equilibrated_state.xml'

    if equil_pdb.exists() and equil_state.exists():
        print(f'[{i+1}/{len(designs_to_equil)}] {design_name}: equilibrated, skipping')
        equil_results.append({'design_name': design_name, 'status': 'cached'})
        continue

    minimized_pdb_path = design_dir / 'minimized.pdb'
    if not minimized_pdb_path.exists():
        print(f'[{i+1}/{len(designs_to_equil)}] {design_name}: no minimized.pdb, skipping')
        equil_results.append({'design_name': design_name, 'status': 'skipped'})
        continue

    t0 = time.time()

    try:
        print(f'[{i+1}/{len(designs_to_equil)}] Equilibrating: {design_name}')

        pdb = PDBFile(str(minimized_pdb_path))
        forcefield = ForceField(*FORCEFIELD_FILES)
        system = forcefield.createSystem(
            pdb.topology,
            nonbondedMethod=PME,
            nonbondedCutoff=1.0 * unit.nanometers,
            constraints=HBonds,
        )

        add_position_restraints(system, pdb.topology, pdb.positions)

        integrator = mm.LangevinMiddleIntegrator(
            300 * unit.kelvin, 1 / unit.picosecond, TIMESTEP
        )
        simulation = Simulation(pdb.topology, system, integrator,
                                platform, platform_properties)
        simulation.context.setPositions(pdb.positions)
        simulation.context.setVelocitiesToTemperature(300 * unit.kelvin)

        print(f'  NVT: {NVT_STEPS * 0.002:.0f} ps...')
        simulation.step(NVT_STEPS)

        system.addForce(mm.MonteCarloBarostat(1 * unit.atmospheres, 300 * unit.kelvin))
        simulation.context.reinitialize(preserveState=True)

        print(f'  NPT: {NPT_STEPS * 0.002:.0f} ps...')
        simulation.step(NPT_STEPS)

        state = simulation.context.getState(getPositions=True, getVelocities=True)
        positions = state.getPositions()

        with open(equil_pdb, 'w') as f:
            PDBFile.writeFile(pdb.topology, positions, f)

        with open(equil_state, 'w') as f:
            f.write(mm.XmlSerializer.serialize(state))

        elapsed = time.time() - t0
        print(f'  ✓ Equilibrated ({elapsed:.0f}s)')

        equil_results.append({
            'design_name': design_name,
            'status': 'equilibrated',
            'equil_time_s': round(elapsed, 1),
        })

    except Exception as e:
        elapsed = time.time() - t0
        print(f'  ✗ FAILED: {e} ({elapsed:.0f}s)')
        equil_results.append({'design_name': design_name, 'status': 'failed', 'error': str(e)})

n_eq = sum(1 for r in equil_results if r['status'] in ('equilibrated', 'cached'))
n_fail = sum(1 for r in equil_results if r['status'] == 'failed')
print(f'\nEquilibration: {n_eq} succeeded, {n_fail} failed')

print(f'\n✓ Cell 5 complete')


Equilibrating 25 designs...

[1/25] len25_clusterA_noise0__design_11_0_rank1: equilibrated, skipping
[2/25] len70_clusterA_noise0__design_3_0_rank7: equilibrated, skipping
[3/25] len70_distributed_noise0__design_5_0_rank2: equilibrated, skipping
[4/25] len70_clusterB_noise0__design_17_0_rank0: equilibrated, skipping
[5/25] len100_clusterA_noise0__design_16_0_rank6: equilibrated, skipping
[6/25] len100_distributed_noise0__design_5_0_rank0: equilibrated, skipping
[7/25] len70_clusterA_noise0__design_16_0_rank5: equilibrated, skipping
[8/25] len25_clusterA_noise0__design_15_0_rank0: equilibrated, skipping
[9/25] len100_clusterA_noise0__design_13_0_rank4: equilibrated, skipping
[10/25] len100_clusterB_noise05__design_0_0_rank1: equilibrated, skipping
[11/25] len70_clusterB_noise0__design_15_0_rank2: equilibrated, skipping
[12/25] len70_distributed_noise0__design_10_0_rank3: equilibrated, skipping
[13/25] len100_clusterA_noise05__design_14_0_rank3: equilibrated, skipping
[14/25] len100_clus

In [ ]:
# ============================================================
# Cell 6 — Production MD: 1 ns NPT
# ============================================================
#
# Dependencies: Cell 0, Cell 1, Cell 2, Cell 3, Cell 4, Cell 5
# External files: equilibrated.pdb + equilibrated_state.xml from Cell 5
# Compute: GPU (CUDA preferred, mixed precision). ~60-90 min per design on T4.
# Purpose: Run 1 ns production MD for each equilibrated design.
#   Position restraints are removed.
#   Trajectory frames saved every 10 ps (100 frames total).
#   Checkpoint every 50 ps for restart safety.
# Restart-safe: detects existing DCD frame count and resumes
#   from checkpoint, running only the remaining steps.
# Outputs: production.dcd, production_log.csv, production.pdb (topology)
# ============================================================

import openmm as mm
from openmm.app import (
    PDBFile, ForceField, Simulation, DCDReporter,
    StateDataReporter, CheckpointReporter,
)
from openmm.app import PME, HBonds
import openmm.unit as unit
import mdtraj as md
import time

FORCEFIELD_FILES = ['amber14-all.xml', 'amber14/tip3p.xml']
TIMESTEP = 0.002 * unit.picoseconds
PRODUCTION_NS = 1.0
PRODUCTION_STEPS = int(PRODUCTION_NS * 1e6 / 2)  # 500,000 steps at 2 fs = 1 ns
REPORT_INTERVAL = 5000     # every 10 ps
DCD_INTERVAL = 5000        # every 10 ps -> 100 frames
CHECKPOINT_INTERVAL = 25000  # every 50 ps
EXPECTED_FRAMES = PRODUCTION_STEPS // DCD_INTERVAL  # 100

designs_to_run = [r for r in equil_results if r['status'] in ('equilibrated', 'cached')]
prod_results = []

print(f'Production MD for {len(designs_to_run)} designs ({PRODUCTION_NS} ns each)...\n')

for i, rec in enumerate(designs_to_run):
    design_name = rec['design_name']
    design_dir = OUTPUT_DIR / design_name
    dcd_file = design_dir / 'production.dcd'
    log_file = design_dir / 'production_log.csv'
    topo_pdb = design_dir / 'production.pdb'
    chk_file = design_dir / 'production.chk'

    equil_pdb_path = design_dir / 'equilibrated.pdb'
    equil_state_path = design_dir / 'equilibrated_state.xml'

    if not equil_pdb_path.exists() or not equil_state_path.exists():
        print(f'[{i+1}/{len(designs_to_run)}] {design_name}: missing equilibrated files, skipping')
        prod_results.append({'design_name': design_name, 'status': 'skipped'})
        continue

    # ── Determine how many frames already exist ──
    existing_frames = 0
    if dcd_file.exists() and topo_pdb.exists():
        try:
            existing_traj = md.load(str(dcd_file), top=str(topo_pdb))
            existing_frames = existing_traj.n_frames
            del existing_traj
        except Exception:
            existing_frames = 0

    if existing_frames >= EXPECTED_FRAMES:
        print(f'[{i+1}/{len(designs_to_run)}] {design_name}: '
              f'production complete ({existing_frames} frames), skipping')
        prod_results.append({'design_name': design_name, 'status': 'cached',
                             'n_frames': existing_frames})
        continue

    frames_remaining = EXPECTED_FRAMES - existing_frames
    steps_remaining = frames_remaining * DCD_INTERVAL

    t0 = time.time()

    try:
        if existing_frames > 0:
            print(f'[{i+1}/{len(designs_to_run)}] Resuming: {design_name} '
                  f'({existing_frames}/{EXPECTED_FRAMES} frames done, '
                  f'{steps_remaining} steps remaining)')
        else:
            print(f'[{i+1}/{len(designs_to_run)}] Production: {design_name}')

        pdb = PDBFile(str(equil_pdb_path))
        forcefield = ForceField(*FORCEFIELD_FILES)
        system = forcefield.createSystem(
            pdb.topology,
            nonbondedMethod=PME,
            nonbondedCutoff=1.0 * unit.nanometers,
            constraints=HBonds,
        )
        system.addForce(mm.MonteCarloBarostat(1 * unit.atmospheres, 300 * unit.kelvin))

        integrator = mm.LangevinMiddleIntegrator(
            300 * unit.kelvin, 1 / unit.picosecond, TIMESTEP
        )

        simulation = Simulation(pdb.topology, system, integrator,
                                platform, platform_properties)

        # Load state: checkpoint if resuming, else equilibrated state
        if existing_frames > 0 and chk_file.exists():
            try:
                # Load equilibrated state first for positions, then checkpoint
                with open(equil_state_path) as f:
                    state_xml = f.read()
                simulation.context.setState(mm.XmlSerializer.deserialize(state_xml))
                simulation.loadCheckpoint(str(chk_file))
                print(f'  Resumed from checkpoint')
            except Exception as e:
                print(f'  Checkpoint load failed ({e}), restarting from equilibrated state')
                existing_frames = 0
                steps_remaining = PRODUCTION_STEPS
                with open(equil_state_path) as f:
                    state_xml = f.read()
                simulation.context.setState(mm.XmlSerializer.deserialize(state_xml))
        else:
            with open(equil_state_path) as f:
                state_xml = f.read()
            simulation.context.setState(mm.XmlSerializer.deserialize(state_xml))

        # Save topology PDB (for loading DCD later)
        if not topo_pdb.exists():
            state = simulation.context.getState(getPositions=True)
            with open(topo_pdb, 'w') as f:
                PDBFile.writeFile(pdb.topology, state.getPositions(), f)

        # Add reporters (append if resuming)
        append_mode = existing_frames > 0
        simulation.reporters.append(DCDReporter(str(dcd_file), DCD_INTERVAL,
                                                append=append_mode))
        simulation.reporters.append(StateDataReporter(
            str(log_file), REPORT_INTERVAL,
            step=True, time=True,
            potentialEnergy=True, temperature=True, density=True,
            speed=True, separator=',',
            append=append_mode,
        ))
        simulation.reporters.append(CheckpointReporter(str(chk_file), CHECKPOINT_INTERVAL))

        # Run only remaining steps
        simulation.step(steps_remaining)

        elapsed = time.time() - t0
        speed_ns_day = (PRODUCTION_NS / (elapsed / 86400)) if elapsed > 0 else 0
        print(f'  ✓ {PRODUCTION_NS} ns complete ({elapsed:.0f}s, '
              f'{speed_ns_day:.1f} ns/day)')

        prod_results.append({
            'design_name': design_name,
            'status': 'complete',
            'production_time_s': round(elapsed, 1),
            'speed_ns_day': round(speed_ns_day, 1),
        })

    except Exception as e:
        elapsed = time.time() - t0
        print(f'  ✗ FAILED after {elapsed:.0f}s: {e}')
        prod_results.append({'design_name': design_name, 'status': 'failed',
                             'error': str(e)})

n_done = sum(1 for r in prod_results if r['status'] in ('complete', 'cached'))
n_fail = sum(1 for r in prod_results if r['status'] == 'failed')
print(f'\n=== Production MD summary ===')
print(f'Complete: {n_done}')
print(f'Failed:   {n_fail}')

df_prod = pd.DataFrame(prod_results)
df_prod.to_csv(OUTPUT_DIR / '10_production_progress.csv', index=False)

print(f'\n✓ Cell 6 complete')


Production MD for 25 designs (1.0 ns each)...

[1/25] len25_clusterA_noise0__design_11_0_rank1: production complete (100 frames), skipping
[2/25] len70_clusterA_noise0__design_3_0_rank7: production complete (100 frames), skipping
[3/25] len70_distributed_noise0__design_5_0_rank2: production complete (100 frames), skipping
[4/25] len70_clusterB_noise0__design_17_0_rank0: production complete (100 frames), skipping
[5/25] len100_clusterA_noise0__design_16_0_rank6: production complete (100 frames), skipping
[6/25] len100_distributed_noise0__design_5_0_rank0: production complete (100 frames), skipping
[7/25] len70_clusterA_noise0__design_16_0_rank5: production complete (100 frames), skipping
[8/25] len25_clusterA_noise0__design_15_0_rank0: production complete (100 frames), skipping
[9/25] len100_clusterA_noise0__design_13_0_rank4: production complete (100 frames), skipping
[10/25] len100_clusterB_noise05__design_0_0_rank1: production complete (100 frames), skipping
[11/25] len70_clusterB_no

In [ ]:
# ============================================================
# Cell 7 — Static scoring on minimized structures
# ============================================================
#
# Dependencies: Cell 0, Cell 1, Cell 4
# External files:
#   - minimized.pdb from Cell 4 (new designs, on Drive)
#   - Existing NB08 PDBs (minimized/equilibrated/production)
#   - Original threaded complex PDBs (fallback)
# Compute: CPU only (no GPU needed)
# Purpose: Compute post-minimization structural interface metrics
#   for ALL validated designs (not just newly minimized ones).
# Metrics:
#   - BSA (BioPython Shrake-Rupley, protein residues only)
#   - Inter-chain residue contact pairs (heavy atoms, 4.5 Å)
#   - Polar close-contacts (N/O/S pairs within 3.5 Å;
#     distance-only proxy, NOT geometry-validated H-bonds)
# Outputs: 10_static_scores.csv with structure_source column
# ============================================================

from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='Bio')

parser = PDBParser(QUIET=True)

CONTACT_CUTOFF = 4.5  # Angstrom
POLAR_CUTOFF = 3.5    # Angstrom, polar heavy-atom distance
POLAR_ELEMENTS = {'N', 'O', 'S'}


def find_best_structure(design_name):
    """Find the best available structure for static scoring.
    Returns (path, source_label) or (None, 'missing')."""

    # Priority 1: new minimized structure from this notebook
    new_min = OUTPUT_DIR / design_name / 'minimized.pdb'
    if new_min.exists():
        return str(new_min), 'nb10_minimized'

    # Priority 2: existing NB08 structures (prefer production topo > equil > min)
    nb08_dir = EXISTING_MD_DIR / design_name
    if nb08_dir.exists():
        for fname in ['production.pdb', 'equilibrated.pdb', 'minimized.pdb']:
            candidate = nb08_dir / fname
            if candidate.exists():
                return str(candidate), f'nb08_{fname.replace(".pdb", "")}'
        # Fall back to any PDB in the directory
        pdbs = sorted(nb08_dir.glob('*.pdb'))
        if pdbs:
            return str(pdbs[0]), 'nb08_other'

    # Priority 3: original threaded complex (unminimized)
    row = df_validated[df_validated['design_name'] == design_name]
    if len(row) > 0 and pd.notna(row['complex_pdb'].values[0]):
        return row['complex_pdb'].values[0], 'threaded_unminimized'

    return None, 'missing'


def compute_bsa_protein_only(structure, chain_ids):
    """Compute BSA using only standard amino acid residues."""
    from Bio.PDB.SASA import ShrakeRupley
    from Bio.PDB.Structure import Structure as BioStructure
    from Bio.PDB.Model import Model
    from Bio.PDB.Chain import Chain as BioChain

    sr = ShrakeRupley()

    # Build a protein-only copy of the structure
    def make_protein_structure(chains_to_include):
        tmp = BioStructure('tmp')
        tmp_model = Model(0)
        for cid in chains_to_include:
            orig_chain = structure[0][cid]
            new_chain = BioChain(cid)
            for res in orig_chain.get_residues():
                if is_aa(res, standard=True):
                    new_chain.add(res.copy())
            tmp_model.add(new_chain)
        tmp.add(tmp_model)
        return tmp

    # Complex SASA (protein only)
    complex_struct = make_protein_structure(chain_ids)
    sr.compute(complex_struct[0], level='R')
    sasa_complex = sum(r.sasa for r in complex_struct[0].get_residues())

    # Individual chain SASAs
    sasa_chains = 0.0
    for cid in chain_ids:
        chain_struct = make_protein_structure([cid])
        sr.compute(chain_struct[0], level='R')
        sasa_chains += sum(r.sasa for r in chain_struct[0].get_residues())

    bsa = (sasa_chains - sasa_complex) / 2.0
    return bsa


def count_contacts_and_polar(structure, chain_a_id, chain_b_id):
    """Count inter-chain contacts and polar close-contact candidates.
    Uses only standard amino acid residues.
    Returns: (n_residue_pairs, n_polar_close_contacts, contacted_target_residues)"""
    chain_a = structure[0][chain_a_id]
    chain_b = structure[0][chain_b_id]

    # Filter to protein heavy atoms
    atoms_a = [(a, a.get_vector().get_array()) for r in chain_a.get_residues()
               if is_aa(r, standard=True)
               for a in r.get_atoms() if a.element != 'H']
    atoms_b = [(a, a.get_vector().get_array()) for r in chain_b.get_residues()
               if is_aa(r, standard=True)
               for a in r.get_atoms() if a.element != 'H']

    if not atoms_a or not atoms_b:
        return 0, 0, set()

    coords_a = np.array([c for _, c in atoms_a])
    coords_b = np.array([c for _, c in atoms_b])
    tree_b = cKDTree(coords_b)

    contact_residue_pairs = set()
    polar_close_contacts = 0
    contacted_target_residues = set()

    for idx_a, (atom_a, coord_a) in enumerate(atoms_a):
        neighbors = tree_b.query_ball_point(coord_a, CONTACT_CUTOFF)
        for j in neighbors:
            atom_b = atoms_b[j][0]
            res_a_id = (chain_a_id, atom_a.get_parent().get_id()[1])
            res_b_id = (chain_b_id, atom_b.get_parent().get_id()[1])
            contact_residue_pairs.add((res_a_id, res_b_id))
            contacted_target_residues.add(res_b_id[1])

            # Polar close-contact check (distance-only, no angle)
            if (atom_a.element in POLAR_ELEMENTS and
                atom_b.element in POLAR_ELEMENTS):
                dist = np.linalg.norm(coord_a - atoms_b[j][1])
                if dist <= POLAR_CUTOFF:
                    polar_close_contacts += 1

    return len(contact_residue_pairs), polar_close_contacts, contacted_target_residues


# ── Score ALL validated designs ──
static_records = []

print(f'Static scoring on {len(ALL_DESIGN_IDS)} validated designs...\n')

for i, design_name in enumerate(ALL_DESIGN_IDS):
    pdb_path, structure_source = find_best_structure(design_name)

    if pdb_path is None:
        print(f'  {design_name}: no structure found, skipping')
        static_records.append({
            'design_name': design_name,
            'structure_source': 'missing',
        })
        continue

    try:
        structure = parser.get_structure(design_name, pdb_path)
        chains = list(structure[0].get_chains())
        chain_ids = [c.id for c in chains]

        # Identify protein chains (skip water/ion chains)
        protein_chains = []
        for c in chains:
            protein_res = [r for r in c.get_residues() if is_aa(r, standard=True)]
            if len(protein_res) > 0:
                protein_chains.append(c.id)

        if len(protein_chains) < 2:
            print(f'  {design_name}: only {len(protein_chains)} protein chain(s), skipping')
            static_records.append({
                'design_name': design_name,
                'structure_source': structure_source,
                'issues': f'only {len(protein_chains)} protein chains',
            })
            continue

        target_chain = protein_chains[0]  # A = PD-L1
        binder_chain = protein_chains[1]  # B = binder

        bsa = compute_bsa_protein_only(structure, [target_chain, binder_chain])
        n_contacts, n_polar, contacted_res = count_contacts_and_polar(
            structure, target_chain, binder_chain
        )

        binder_residues = len([r for r in structure[0][binder_chain].get_residues()
                               if is_aa(r, standard=True)])

        static_records.append({
            'design_name': design_name,
            'structure_source': structure_source,
            'bsa_minimized': round(bsa, 1),
            'contact_residue_pairs': n_contacts,
            'polar_close_contacts': n_polar,
            'contacted_target_residues': len(contacted_res),
            'binder_length': binder_residues,
            'bsa_per_binder_residue': round(bsa / max(binder_residues, 1), 1),
        })

        if (i + 1) % 5 == 0 or i == 0:
            print(f'  [{i+1}/{len(ALL_DESIGN_IDS)}] {design_name} ({structure_source}): '
                  f'BSA={bsa:.0f}, contacts={n_contacts}, polar={n_polar}')

    except Exception as e:
        print(f'  {design_name}: ERROR - {e}')
        static_records.append({
            'design_name': design_name,
            'structure_source': structure_source,
            'issues': str(e),
        })

# Ensure expected columns exist even if all fail
STATIC_COLUMNS = ['design_name', 'structure_source', 'bsa_minimized',
                  'contact_residue_pairs', 'polar_close_contacts',
                  'contacted_target_residues', 'binder_length',
                  'bsa_per_binder_residue']
df_static = pd.DataFrame(static_records)
for col in STATIC_COLUMNS:
    if col not in df_static.columns:
        df_static[col] = np.nan

static_csv = OUTPUT_DIR / '10_static_scores.csv'
df_static.to_csv(static_csv, index=False)

n_scored = df_static['bsa_minimized'].notna().sum()
print(f'\n✓ Saved: {static_csv} ({n_scored}/{len(df_static)} designs scored)')
print(f'  Sources: {df_static["structure_source"].value_counts().to_dict()}')

print(f'\n✓ Cell 7 complete')


In [ ]:
# ============================================================
# Cell 8 — PRODIGY binding affinity prediction
# ============================================================
#
# Dependencies: Cell 0, Cell 1, Cell 7
# External files: structures identified in Cell 7
# Compute: CPU only (no GPU needed)
# Purpose: Run PRODIGY on available structures to estimate
#   relative dG and Kd for each design.
#
# IMPORTANT CAVEATS:
#   - PRODIGY was trained on natural protein-protein complexes.
#   - These are de novo designed interfaces, not natural complexes.
#   - Absolute Kd estimates should NOT be trusted.
#   - Relative rankings within this design set MAY be informative.
#   - PRODIGY r ~ 0.73 with experimental affinity on its benchmark;
#     performance on designed interfaces is unknown.
#
# Outputs: 10_prodigy_scores.csv
# ============================================================

import subprocess
import re

PRODIGY_COLUMNS = ['design_name', 'prodigy_status',
                   'prodigy_dG_kcal_mol', 'prodigy_Kd_M']

prodigy_records = []

# Score all designs that have static structures
scoreable = df_static[df_static['bsa_minimized'].notna()].copy()

print(f'Running PRODIGY on {len(scoreable)} structures...\n')

for i, (_, row) in enumerate(scoreable.iterrows()):
    design_name = row['design_name']

    # Find the structure (same logic as Cell 7)
    pdb_path, _ = find_best_structure(design_name)
    if pdb_path is None:
        continue

    try:
        result = subprocess.run(
            [sys.executable, '-m', 'prodigy', str(pdb_path),
             '--selection', 'A', 'B'],
            capture_output=True, text=True, timeout=120,
        )

        output = result.stdout + result.stderr

        dg_match = re.search(r'Predicted binding affinity.*?(-?\d+\.\d+)', output)
        kd_match = re.search(r'Predicted dissociation constant.*?(\d+\.\d+[eE][+-]?\d+)', output)

        dg = float(dg_match.group(1)) if dg_match else None
        kd = float(kd_match.group(1)) if kd_match else None

        prodigy_records.append({
            'design_name': design_name,
            'prodigy_dG_kcal_mol': dg,
            'prodigy_Kd_M': kd,
            'prodigy_status': 'success' if dg is not None else 'parse_error',
        })

        if (i + 1) % 5 == 0 or i == 0:
            dg_str = f'{dg:.1f}' if dg is not None else '?'
            kd_str = f'{kd:.2e}' if kd is not None else '?'
            print(f'  [{i+1}/{len(scoreable)}] {design_name}: '
                  f'dG={dg_str} kcal/mol, Kd={kd_str} M')

    except subprocess.TimeoutExpired:
        print(f'  {design_name}: PRODIGY timed out')
        prodigy_records.append({
            'design_name': design_name, 'prodigy_status': 'timeout'})
    except Exception as e:
        print(f'  {design_name}: ERROR - {e}')
        prodigy_records.append({
            'design_name': design_name, 'prodigy_status': f'error: {e}'})

# Ensure expected columns even if all fail
if len(prodigy_records) == 0:
    df_prodigy = pd.DataFrame(columns=PRODIGY_COLUMNS)
else:
    df_prodigy = pd.DataFrame(prodigy_records)
    for col in PRODIGY_COLUMNS:
        if col not in df_prodigy.columns:
            df_prodigy[col] = np.nan

prodigy_csv = OUTPUT_DIR / '10_prodigy_scores.csv'
df_prodigy.to_csv(prodigy_csv, index=False)

n_success = (df_prodigy['prodigy_status'] == 'success').sum() if len(df_prodigy) > 0 else 0
print(f'\nPRODIGY complete: {n_success}/{len(df_prodigy)} scored')

if n_success > 0:
    scored = df_prodigy[df_prodigy['prodigy_status'] == 'success']
    print(f'  dG range: {scored["prodigy_dG_kcal_mol"].min():.1f} to '
          f'{scored["prodigy_dG_kcal_mol"].max():.1f} kcal/mol')

print(f'\n✓ Saved: {prodigy_csv}')
print(f'\n✓ Cell 8 complete')


In [ ]:
# ============================================================
# Cell 9 — MD-derived stability scoring
# ============================================================
#
# Dependencies: Cell 0, Cell 1, Cell 6
# External files:
#   - production.dcd + production.pdb from Cell 6 (new sims)
#   - Existing MD data from NB08 (in md_simulations_threaded_complexes/)
# Compute: CPU only (no GPU needed)
# Purpose: Compute trajectory-derived stability metrics.
# Metrics:
#   - Binder backbone RMSD (vs frame 0)
#   - Target backbone RMSD (vs frame 0)
#   - Binder-target COM distance and drift
#   - Binder heavy atoms in contact per frame
#     (NOTE: this is an atom-level metric, not directly comparable
#     to the residue-pair contacts in Cell 7)
#   - Binder Ca RMSF
#
# Uses manual minimum-image PBC correction from NB08b.
# Outputs: 10_md_stability_scores.csv
# ============================================================

import mdtraj as md
from scipy.spatial import cKDTree

CONTACT_CUTOFF_NM = 0.45  # 4.5 Angstrom in nm

MD_COLUMNS = ['design_name', 'md_source', 'n_frames',
              'binder_rmsd_mean', 'binder_rmsd_final',
              'target_rmsd_mean', 'target_rmsd_final',
              'com_dist_mean', 'com_dist_final', 'com_dist_drift',
              'binder_atoms_in_contact_mean', 'binder_atoms_in_contact_final',
              'binder_rmsf_mean', 'binder_rmsf_max']


def minimum_image_correction(traj, binder_idx, target_idx):
    """Apply minimum-image correction per frame for PBC artifacts."""
    corrected_xyz = traj.xyz.copy()
    for frame in range(traj.n_frames):
        if traj.unitcell_vectors is None:
            continue
        box = traj.unitcell_vectors[frame]
        com_binder = corrected_xyz[frame, binder_idx].mean(axis=0)
        com_target = corrected_xyz[frame, target_idx].mean(axis=0)
        delta = com_binder - com_target
        for dim in range(3):
            box_len = np.linalg.norm(box[dim])
            shift = np.round(delta[dim] / box_len)
            if shift != 0:
                corrected_xyz[frame, binder_idx] -= shift * box[dim]
    return corrected_xyz


def analyze_trajectory(design_name, topo_pdb, dcd_path):
    """Compute all MD-derived stability metrics for one design."""
    traj = md.load(str(dcd_path), top=str(topo_pdb))

    target_idx = traj.topology.select('chainid 0')
    binder_idx = traj.topology.select('chainid 1')
    target_bb = traj.topology.select('chainid 0 and backbone')
    binder_bb = traj.topology.select('chainid 1 and backbone')

    # PBC correction
    corrected_xyz = minimum_image_correction(traj, binder_idx, target_idx)
    traj_c = traj[:]
    traj_c.xyz = corrected_xyz

    # Binder backbone RMSD
    traj_binder = traj_c.atom_slice(binder_bb)
    binder_rmsd = md.rmsd(traj_binder, traj_binder, frame=0) * 10  # nm -> A

    # Target backbone RMSD
    traj_target = traj_c.atom_slice(target_bb)
    target_rmsd = md.rmsd(traj_target, traj_target, frame=0) * 10

    # COM distance
    com_b = np.array([corrected_xyz[f, binder_idx].mean(axis=0) for f in range(traj.n_frames)])
    com_t = np.array([corrected_xyz[f, target_idx].mean(axis=0) for f in range(traj.n_frames)])
    com_dist = np.linalg.norm(com_b - com_t, axis=1) * 10

    # Interface contacts per frame (atom-level: binder heavy atoms with any target neighbor)
    heavy_binder = traj.topology.select('chainid 1 and not element H')
    heavy_target = traj.topology.select('chainid 0 and not element H')
    contact_counts = []
    for f in range(traj.n_frames):
        coords_b = corrected_xyz[f, heavy_binder]
        coords_t = corrected_xyz[f, heavy_target]
        tree = cKDTree(coords_t)
        hits = tree.query_ball_point(coords_b, CONTACT_CUTOFF_NM)
        n_contacts = sum(1 for pts in hits if len(pts) > 0)
        contact_counts.append(n_contacts)

    # Binder RMSF
    binder_ca = traj.topology.select('chainid 1 and name CA')
    traj_bca = traj_c.atom_slice(binder_ca)
    rmsf = md.rmsf(traj_bca, traj_bca, frame=0) * 10

    return {
        'design_name': design_name,
        'n_frames': traj.n_frames,
        'binder_rmsd_mean': round(float(binder_rmsd.mean()), 2),
        'binder_rmsd_final': round(float(binder_rmsd[-1]), 2),
        'target_rmsd_mean': round(float(target_rmsd.mean()), 2),
        'target_rmsd_final': round(float(target_rmsd[-1]), 2),
        'com_dist_mean': round(float(com_dist.mean()), 2),
        'com_dist_final': round(float(com_dist[-1]), 2),
        'com_dist_drift': round(float(com_dist[-1] - com_dist[0]), 2),
        'binder_atoms_in_contact_mean': round(float(np.mean(contact_counts)), 1),
        'binder_atoms_in_contact_final': int(contact_counts[-1]),
        'binder_rmsf_mean': round(float(rmsf.mean()), 2),
        'binder_rmsf_max': round(float(rmsf.max()), 2),
    }


def find_nb08_trajectory(design_name):
    """Deterministic file selection for existing NB08 trajectories.
    Returns (topo_path, dcd_path) or (None, None)."""
    existing_dir = EXISTING_MD_DIR / design_name
    if not existing_dir.exists():
        return None, None

    # Prefer specific topology files in priority order
    topo = None
    for fname in ['production.pdb', 'production_topology.pdb',
                  'trajectory_topology.pdb', 'equilibrated.pdb',
                  'minimized.pdb']:
        candidate = existing_dir / fname
        if candidate.exists():
            topo = candidate
            break

    # Fall back: if only one PDB, use it; if multiple, skip
    if topo is None:
        pdbs = sorted(existing_dir.glob('*.pdb'))
        if len(pdbs) == 1:
            topo = pdbs[0]
        elif len(pdbs) > 1:
            print(f'  WARNING: {design_name} has {len(pdbs)} PDBs, '
                  f'cannot determine topology automatically')
            return None, None

    # DCD: prefer production.dcd, else single DCD
    dcd = existing_dir / 'production.dcd'
    if not dcd.exists():
        dcds = sorted(existing_dir.glob('*.dcd'))
        if len(dcds) == 1:
            dcd = dcds[0]
        else:
            return None, None

    if topo is None:
        return None, None

    return topo, dcd


# ── Analyze all trajectories ──
md_records = []

# New simulations from Cell 6
for rec in prod_results:
    if rec['status'] not in ('complete', 'cached'):
        continue
    design_name = rec['design_name']
    design_dir = OUTPUT_DIR / design_name
    topo = design_dir / 'production.pdb'
    dcd = design_dir / 'production.dcd'
    if not topo.exists() or not dcd.exists():
        continue
    try:
        print(f'Analyzing: {design_name}')
        result = analyze_trajectory(design_name, topo, dcd)
        result['md_source'] = 'nb10'
        md_records.append(result)
    except Exception as e:
        print(f'  ERROR: {e}')

# Existing simulations from NB08
for design_name in sorted(existing_md_designs):
    topo, dcd = find_nb08_trajectory(design_name)
    if topo is None or dcd is None:
        print(f'Existing MD {design_name}: could not resolve topo/dcd, skipping')
        continue

    try:
        print(f'Analyzing existing: {design_name} (topo={topo.name}, dcd={dcd.name})')
        # Validate atom count match
        traj_test = md.load(str(dcd), top=str(topo))
        print(f'  Loaded: {traj_test.n_frames} frames, {traj_test.n_atoms} atoms')
        del traj_test

        result = analyze_trajectory(design_name, topo, dcd)
        result['md_source'] = 'nb08'
        md_records.append(result)
    except Exception as e:
        print(f'  ERROR: {e}')

# Ensure expected columns even if no trajectories
if len(md_records) == 0:
    df_md = pd.DataFrame(columns=MD_COLUMNS)
else:
    df_md = pd.DataFrame(md_records)
    for col in MD_COLUMNS:
        if col not in df_md.columns:
            df_md[col] = np.nan

md_csv = OUTPUT_DIR / '10_md_stability_scores.csv'
df_md.to_csv(md_csv, index=False)
print(f'\n✓ Saved: {md_csv} ({len(df_md)} designs scored)')

print(f'\n✓ Cell 9 complete')


In [ ]:
# ============================================================
# Cell 10 — Consensus scorecard assembly
# ============================================================
#
# Dependencies: Cell 0, Cell 1, Cell 7, Cell 8, Cell 9
# External files: CSVs from Cells 7-9
# Compute: CPU only (no GPU needed)
# Purpose: Merge static scores, PRODIGY scores, MD stability
#   scores, Boltz-2 confidence metrics, and fingerprinting
#   features into a single consensus scorecard.
#   Adds evidence flags showing which data is available per design.
# Outputs:
#   - 10_complex_scorecard.csv (all features + evidence flags)
#   - 10_consensus_ranking.csv (sorted by composite score)
# ============================================================

# ── Load all scoring tables ──
df_static = pd.read_csv(OUTPUT_DIR / '10_static_scores.csv')
df_prodigy = pd.read_csv(OUTPUT_DIR / '10_prodigy_scores.csv')
df_md = pd.read_csv(OUTPUT_DIR / '10_md_stability_scores.csv')

print(f'Static scores:  {len(df_static)} designs')
print(f'PRODIGY scores: {len(df_prodigy)} designs')
print(f'MD scores:      {len(df_md)} designs')

# ── Start from master table ──
scorecard = df_validated[['design_name', 'boltz_tier']].copy()

for col in ['ipTM', 'complex_pLDDT', 'pTM', 'hotspot_config', 'length']:
    if col in df_validated.columns:
        scorecard[col] = df_validated[col].values

if 'fp_alignment_rmsd' in df_validated.columns:
    scorecard['fp_alignment_rmsd'] = df_validated['fp_alignment_rmsd'].values

# ── Merge scoring tables ──
# Static: include structure_source
static_merge_cols = ['design_name', 'structure_source', 'bsa_minimized',
                     'contact_residue_pairs', 'polar_close_contacts',
                     'contacted_target_residues', 'binder_length',
                     'bsa_per_binder_residue']
static_merge_cols = [c for c in static_merge_cols if c in df_static.columns]
scorecard = scorecard.merge(df_static[static_merge_cols], on='design_name', how='left')

# PRODIGY
prodigy_merge_cols = ['design_name', 'prodigy_dG_kcal_mol', 'prodigy_Kd_M']
prodigy_merge_cols = [c for c in prodigy_merge_cols if c in df_prodigy.columns]
if len(prodigy_merge_cols) > 1:
    scorecard = scorecard.merge(df_prodigy[prodigy_merge_cols], on='design_name', how='left')

# MD
md_merge_cols = ['design_name', 'md_source', 'binder_rmsd_mean', 'binder_rmsd_final',
                 'target_rmsd_mean', 'com_dist_drift',
                 'binder_atoms_in_contact_mean', 'binder_rmsf_mean', 'binder_rmsf_max']
md_merge_cols = [c for c in md_merge_cols if c in df_md.columns]
if len(md_merge_cols) > 1:
    scorecard = scorecard.merge(df_md[md_merge_cols], on='design_name', how='left')

# ── Evidence flags ──
scorecard['has_static'] = scorecard['bsa_minimized'].notna()
scorecard['has_prodigy'] = scorecard.get('prodigy_dG_kcal_mol', pd.Series(dtype=float)).notna()
scorecard['has_md'] = scorecard.get('binder_rmsd_mean', pd.Series(dtype=float)).notna()

# Score confidence: high = all three, medium = static + one of prodigy/md, low = static only or less
def assign_confidence(row):
    n_sources = sum([row.get('has_static', False),
                     row.get('has_prodigy', False),
                     row.get('has_md', False)])
    if n_sources >= 3:
        return 'high'
    elif n_sources >= 2:
        return 'medium'
    else:
        return 'low'

scorecard['score_confidence'] = scorecard.apply(assign_confidence, axis=1)

print(f'\nScorecard: {len(scorecard)} designs x {len(scorecard.columns)} features')
print(f'Evidence: {scorecard["score_confidence"].value_counts().to_dict()}')

# ── Save full scorecard ──
scorecard_csv = OUTPUT_DIR / '10_complex_scorecard.csv'
scorecard.to_csv(scorecard_csv, index=False)
print(f'\n✓ Saved: {scorecard_csv}')

# ── Consensus ranking ──
# FIXED rank_normalize: higher normalized score = better design
def rank_normalize(series, higher_is_better=True):
    """Rank-normalize to 0-1 where 1.0 = best in set.
    higher_is_better=True: highest raw value gets 1.0
    higher_is_better=False: lowest raw value gets 1.0"""
    valid = series.dropna()
    out = pd.Series(np.nan, index=series.index)
    if len(valid) <= 1:
        out.loc[valid.index] = 0.5  # singleton gets middle score
        return out
    ranks = valid.rank(ascending=not higher_is_better)
    norm = (ranks - 1) / (len(ranks) - 1)
    out.loc[valid.index] = norm
    return out

ranking = scorecard.copy()

# Static metrics (larger = better)
ranking['score_bsa'] = rank_normalize(ranking['bsa_minimized'], higher_is_better=True)
ranking['score_contacts'] = rank_normalize(ranking.get('contact_residue_pairs',
    pd.Series(dtype=float)), higher_is_better=True)
ranking['score_polar'] = rank_normalize(ranking.get('polar_close_contacts',
    pd.Series(dtype=float)), higher_is_better=True)

# Boltz-2 ipTM (higher = better)
if 'ipTM' in ranking.columns:
    ranking['score_boltz'] = rank_normalize(ranking['ipTM'], higher_is_better=True)

# PRODIGY dG (more negative = better binding, so lower numerical value = better)
if 'prodigy_dG_kcal_mol' in ranking.columns:
    ranking['score_prodigy'] = rank_normalize(
        ranking['prodigy_dG_kcal_mol'], higher_is_better=False
    )

# MD stability (lower RMSD = better)
if 'binder_rmsd_mean' in ranking.columns:
    ranking['score_stability'] = rank_normalize(
        ranking['binder_rmsd_mean'], higher_is_better=False
    )

# COM drift (less drift = better, use absolute value)
if 'com_dist_drift' in ranking.columns:
    ranking['score_engagement'] = rank_normalize(
        ranking['com_dist_drift'].abs(), higher_is_better=False
    )

# ── Composite scores ──
# Static-only composite (all designs)
static_score_cols = [c for c in ['score_bsa', 'score_contacts', 'score_polar', 'score_boltz']
                     if c in ranking.columns]
if 'score_prodigy' in ranking.columns:
    static_score_cols.append('score_prodigy')
ranking['composite_static'] = ranking[static_score_cols].mean(axis=1)

# Full composite (static + MD where available)
full_score_cols = static_score_cols.copy()
for mc in ['score_stability', 'score_engagement']:
    if mc in ranking.columns:
        full_score_cols.append(mc)
ranking['composite_full'] = ranking[full_score_cols].mean(axis=1)

# Sort descending: higher composite = better design
ranking = ranking.sort_values('composite_full', ascending=False)
ranking['consensus_rank'] = range(1, len(ranking) + 1)

# ── Display top 10 ──
display_cols = ['consensus_rank', 'design_name', 'hotspot_config',
                'score_confidence',
                'bsa_minimized', 'contact_residue_pairs', 'polar_close_contacts',
                'prodigy_dG_kcal_mol',
                'binder_rmsd_mean', 'com_dist_drift',
                'composite_full']
display_cols = [c for c in display_cols if c in ranking.columns]

print('\n=== Top 10 designs by consensus score ===')
print(ranking[display_cols].head(10).to_string(index=False))

ranking_csv = OUTPUT_DIR / '10_consensus_ranking.csv'
ranking.to_csv(ranking_csv, index=False)
print(f'\n✓ Saved: {ranking_csv}')

print(f'\n✓ Cell 10 complete')


In [ ]:
# ============================================================
# Cell 11 — Final summary and structured output
# ============================================================
#
# Dependencies: Cell 0, Cell 1, Cell 7, Cell 8, Cell 9, Cell 10
# External files: CSVs from previous cells
# Compute: CPU only (no GPU needed)
# Purpose: Generate structured JSON summary for portfolio page.
# Outputs: 10_scoring_summary.json
# ============================================================

import json

summary = {
    'notebook': '10_affinity_and_complex_stability_scoring',
    'n_designs_scored': len(scorecard),
    'n_designs_with_static': int(scorecard['has_static'].sum()),
    'n_designs_with_prodigy': int(scorecard['has_prodigy'].sum()),
    'n_designs_with_md': int(scorecard['has_md'].sum()),
    'confidence_distribution': scorecard['score_confidence'].value_counts().to_dict(),
    'scoring_methods': {
        'static': [
            'BSA (protein-only, Shrake-Rupley)',
            'residue_contact_pairs (4.5A heavy-atom cutoff)',
            'polar_close_contacts (3.5A N/O/S distance-only proxy, not geometry-validated H-bonds)',
        ],
        'affinity_proxy': ['PRODIGY_dG', 'PRODIGY_Kd'],
        'md_stability': [
            'binder_backbone_RMSD',
            'target_backbone_RMSD',
            'COM_drift',
            'binder_atoms_in_contact (atom-level, not residue-pair)',
            'binder_Ca_RMSF',
        ],
        'independent_validation': ['Boltz2_ipTM', 'Boltz2_pLDDT'],
    },
    'simulation_params': {
        'forcefield': 'AMBER14',
        'water_model': 'TIP3P',
        'ionic_strength': '150 mM NaCl',
        'production_length_ns': PRODUCTION_NS,
        'temperature_K': 300,
        'pressure_atm': 1,
        'gpu_precision': 'mixed',
    },
}

# Top 5
top5 = ranking.head(5)
summary['top_designs'] = []
for _, row in top5.iterrows():
    entry = {
        'rank': int(row['consensus_rank']),
        'design_name': row['design_name'],
        'composite_score': round(float(row['composite_full']), 3)
            if pd.notna(row['composite_full']) else None,
        'score_confidence': row.get('score_confidence', 'unknown'),
    }
    for col in ['hotspot_config', 'bsa_minimized', 'contact_residue_pairs',
                'ipTM', 'prodigy_dG_kcal_mol', 'binder_rmsd_mean']:
        if col in row.index and pd.notna(row[col]):
            val = row[col]
            entry[col] = round(float(val), 3) if isinstance(val, (float, np.floating)) else val
    summary['top_designs'].append(entry)

# Score distributions
for metric, key in [('prodigy_dG_kcal_mol', 'prodigy_stats'),
                    ('bsa_minimized', 'bsa_stats')]:
    if metric in scorecard.columns:
        vals = scorecard[metric].dropna()
        if len(vals) > 0:
            summary[key] = {
                'mean': round(float(vals.mean()), 2),
                'std': round(float(vals.std()), 2),
                'min': round(float(vals.min()), 2),
                'max': round(float(vals.max()), 2),
            }

# Caveats
summary['caveats'] = [
    'Short MD (1 ns) does not prove binding.',
    'PRODIGY dG estimates are approximate and trained on natural complexes.',
    'Absolute Kd estimates should not be trusted for de novo designed interfaces.',
    'Polar close-contacts are distance-only proxies, not geometry-validated H-bonds.',
    'Static residue-pair contacts and MD atom-level contacts are different metrics.',
    'Scores are useful for ranking and workflow development, not validation.',
    'Designs with low score_confidence have fewer evidence sources.',
]

summary_path = OUTPUT_DIR / '10_scoring_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=lambda o: float(o) if hasattr(o, 'item') else str(o))

print(json.dumps(summary, indent=2, default=lambda o: float(o) if hasattr(o, 'item') else str(o)))

print(f'\n=== Output files ===')
for p in sorted(OUTPUT_DIR.glob('10_*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'  {p.name} ({size_kb:.1f} KB)')

print(f'\n=== Interpretation ===')
print('These scores are NOT experimental affinities.')
print('They are comparative triage features. The consensus ranking')
print('prioritizes designs for experimental testing. Top-ranked')
print('designs are hypotheses, not validated binders.')
print(f'\nDesigns with "high" confidence have static, PRODIGY, and MD evidence.')
print(f'Designs with "low" confidence should be interpreted cautiously.')

print(f'\n✓ Notebook 10 complete')
